# Exploratory Data Analysis (EDA)

## Objective

Before estimating the causal impact of the marketing campaign, it is important to understand the dataset and assess whether the treatment and control groups are comparable.

This notebook explores customer characteristics, treatment assignment, and outcome distributions to identify potential sources of selection bias.

# Dataset Overview

The dataset contains customer demographic information, historical purchasing behavior, marketing treatment assignment, and campaign outcomes.

Key variables include:

| Variable | Description |
|----------|-------------|
| history | Customer's historical spending |
| recency | Months since last purchase |
| channel | Acquisition channel |
| newbie | New customer indicator |
| zip_code | Customer region |
| mens | Purchased men's products previously |
| womens | Purchased women's products previously |
| treatment | Whether the customer received the marketing email |
| conversion | Whether the customer made a purchase |
| spend | Purchase amount after the campaign |


### Step 1: Import Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px

from scipy.stats import chi2_contingency

Matplotlib is building the font cache; this may take a moment.


### Step 2: Load Dataset

In [3]:
df = pd.read_csv("../data/raw/hillstrom.csv")

df.head()

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


### Step 3: Dataset Overview

In [4]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

df.info()

Rows: 64000
Columns: 12
<class 'pandas.DataFrame'>
RangeIndex: 64000 entries, 0 to 63999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   recency          64000 non-null  int64  
 1   history_segment  64000 non-null  str    
 2   history          64000 non-null  float64
 3   mens             64000 non-null  int64  
 4   womens           64000 non-null  int64  
 5   zip_code         64000 non-null  str    
 6   newbie           64000 non-null  int64  
 7   channel          64000 non-null  str    
 8   segment          64000 non-null  str    
 9   visit            64000 non-null  int64  
 10  conversion       64000 non-null  int64  
 11  spend            64000 non-null  float64
dtypes: float64(2), int64(6), str(4)
memory usage: 5.9 MB


### Step 4: Missing Values

In [5]:
missing = (
    df.isnull()
      .sum()
      .to_frame("Missing Values")
)

missing["Percentage"] = (
    missing["Missing Values"] / len(df) * 100
)

missing.sort_values(
    "Missing Values",
    ascending=False
)

,Missing Values,Percentage
recency,0,0.0
history_segment,0,0.0
history,0,0.0
mens,0,0.0
womens,0,0.0
zip_code,0,0.0
newbie,0,0.0
channel,0,0.0
segment,0,0.0
visit,0,0.0


### Step 5: Descriptive Statistics

In [6]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
recency,64000.0,5.763734,3.507592,1.00,2.00,6.00,9.0000,12.00
history,64000.0,242.085656,256.158608,29.99,64.66,158.11,325.6575,3345.93
mens,64000.0,0.551031,0.497393,0.00,0.00,1.00,1.0000,1.00
womens,64000.0,0.549719,0.497526,0.00,0.00,1.00,1.0000,1.00
newbie,64000.0,0.502250,0.499999,0.00,0.00,1.00,1.0000,1.00
visit,64000.0,0.146781,0.353890,0.00,0.00,0.00,0.0000,1.00
conversion,64000.0,0.009031,0.094604,0.00,0.00,0.00,0.0000,1.00
spend,64000.0,1.050908,15.036448,0.00,0.00,0.00,0.0000,499.00


### Understand the Treatment

Hillstrom dataset contain a segment column like:

No E-Mail
Mens E-Mail
Womens E-Mail

Let's convert it into a binary treatment.

In [7]:
df["treatment"] = (
    df["segment"] != "No E-Mail"
).astype(int)

### Treatment Distribution

In [8]:
treatment_counts = (
    df["treatment"]
    .value_counts()
    .rename({0:"Control",1:"Treatment"})
)

treatment_counts

treatment
Treatment    42694
Control      21306
Name: count, dtype: int64

### Treatment Assignment Summary

The dataset contains 64,000 customers:

- Treatment group: 42,694 customers (66.7%)
- Control group: 21,306 customers (33.3%)

Although the treatment and control groups are not equal in size, this alone does not indicate selection bias. The critical question is whether customers in the two groups differ systematically in their characteristics.

The next step is to compare demographic and behavioral features across the two groups to assess covariate balance before estimating treatment effects.

In [9]:
px.bar(
    treatment_counts,
    title="Treatment vs Control Distribution"
)

### Explore the Outcome

Let's understand conversions.

In [10]:
conversion_rate = (
    df.groupby("treatment")["conversion"]
      .mean()
      .reset_index()
)

conversion_rate

,treatment,conversion
0,0,0.005726
1,1,0.010681


In [11]:
px.bar(
    conversion_rate,
    x="treatment",
    y="conversion",
    title="Conversion Rate by Group"
)

### Initial Conversion Analysis

The treatment group achieved a conversion rate of **1.07%**, while the control group converted at **0.57%**.

This corresponds to:

- **Absolute Lift:** 0.50 percentage points
- **Relative Lift:** 86.5%

At first glance, the marketing campaign appears to have increased customer conversions. However, this comparison is purely descriptive and does not establish causality.

The observed difference may be influenced by systematic differences between customers in the treatment and control groups. Before attributing the increase in conversions to the campaign, we must account for potential confounding factors using causal inference techniques.

Therefore, these results should be interpreted as an initial observation rather than an estimate of the true causal effect.

### Spending

In [12]:
df.groupby("treatment")["spend"].describe()

,count,mean,std,min,25%,50%,75%,max
treatment,,,,,,,,
0,21306.0,0.652789,11.58820,0.0,0.0,0.0,0.0,499.0
1,42694.0,1.249585,16.48625,0.0,0.0,0.0,0.0,499.0


In [14]:
px.box(
    df[df["spend"] > 0],
    x="treatment",
    y="spend",
    points="outliers",
    title="Purchase Amount by Treatment Group (Purchasers Only)"
)

### Customer History

In [16]:
df.groupby("treatment")["history"].describe()

,count,mean,std,min,25%,50%,75%,max
treatment,,,,,,,,
0,21306.0,240.882653,252.739362,29.99,65.30,156.655,325.1675,3345.93
1,42694.0,242.686002,257.848830,29.99,64.35,158.870,325.7800,3215.97


In [15]:
px.histogram(
    df,
    x="history",
    color="treatment",
    marginal="box",
    title="Historical Spending"
)

### Historical Spending

Historical spending is a strong indicator of customer purchasing behavior and one of the most important covariates when evaluating marketing campaigns.

The average historical spending is:

- **Control Group:** \$240.88
- **Treatment Group:** \$242.69

The median spending is also very similar across both groups.

These results suggest that customers in the treatment and control groups had comparable purchasing histories prior to the campaign. This indicates good baseline balance for the `history` variable and reduces concerns that historical spending alone influenced treatment assignment.

While this is encouraging, balance should also be assessed across other customer characteristics before drawing conclusions about the comparability of the two groups.

In [17]:
px.box(
    df,
    x="treatment",
    y="recency",
    title="Purchase Recency"
)

In [18]:
channel = (
    pd.crosstab(
        df["channel"],
        df["treatment"],
        normalize="columns"
    )
)

channel

treatment,0,1
channel,,
Multichannel,0.122313,0.120766
Phone,0.437764,0.437860
Web,0.439923,0.441373


In [19]:
px.bar(
    channel,
    barmode="group"
)

In [20]:
features = [
    "history",
    "recency",
    "mens",
    "womens"
]

for feature in features:
    fig = px.box(
        df,
        x="treatment",
        y=feature,
        title=feature
    )
    fig.show()

In [21]:
numeric = df.select_dtypes(
    include=np.number
)

px.imshow(
    numeric.corr(),
    text_auto=True,
    title="Correlation Matrix"
)

### Standardized Mean Difference (SMD)

In [22]:
def standardized_mean_difference(df, feature, treatment):

    treated = df[df[treatment] == 1][feature]
    control = df[df[treatment] == 0][feature]

    numerator = treated.mean() - control.mean()

    pooled_std = np.sqrt(
        (
            treated.var() +
            control.var()
        ) / 2
    )

    return numerator / pooled_std

In [23]:
continuous = [
    "history",
    "recency",
    "mens",
    "womens"
]

for col in continuous:
    print(
        col,
        standardized_mean_difference(
            df,
            col,
            "treatment"
        )
    )

history 0.0070634545705258595
recency 0.006004340077750869
mens -0.006610559041797893
womens 0.00626509090065978


### Covariate Balance Assessment

To evaluate whether the treatment and control groups are comparable, we computed the Standardized Mean Difference (SMD) for key customer characteristics.

| Feature | SMD |
|---------|----:|
| History | 0.007 |
| Recency | 0.006 |
| Men's Purchases | - 0.007 |
| Women's Purchases | 0.006 |

All covariates have an absolute SMD well below the commonly accepted threshold of **0.10**, indicating excellent covariate balance.

These results suggest that treatment assignment is largely independent of the observed customer characteristics, consistent with the randomized design of the marketing campaign.